# Volume 4, Chapter 6: Fourier Series (Python companion)

Exercises first, solutions at the end. We measure the Gibbs overshoot, compute Fourier coefficients with the FFT (Chapter 7 explains how it works), test equidistribution and Benford's law, apply Weyl's criterion to expose a famous bad random number generator, and check the isoperimetric inequality on a few curves. Requirements: numpy and mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P6.1 (★) Gibbs against Fejér.** For the square wave, compute the maximum of $S_{2N-1}f$ near the jump and the maximum of the Fejér mean $\sigma_{2N}f$ on $]0,\frac\pi2]$, for $N=5,20,100,400$. Compare with Theorem 6.12.

**P6.2 (★) Coefficients by FFT.** Sample $x$ and $x^2$ at $M=2^{12}$ points of $[-\pi,\pi[$ and approximate $c_n$ by the discrete sum $\frac1M\sum_jf(t_j)e^{-int_j}$, computed by `np.fft.fft`. Compare with the exact values of Exercise 6.1 and check Parseval's identity (Theorem 6.11). Why is the error so much larger for $x$ than for $x^2$?

**P6.3 (★★) How uniform is $n\alpha$?** The *discrepancy* $D_N$ of $u_1,\dots,u_N$ is the largest difference, over intervals $[0,a[$, between the proportion of points in the interval and its length. Compute $D_N$ for $\{n\alpha\}$, $N=10^5$, with $\alpha$ the golden ratio, $\sqrt2$, $\pi$, $e$, and for $10^5$ random points. Then test Exercise 6.5 on the first $10\,000$ powers of $2$.

**P6.4 (★★) CS application: the spectral test.** A linear congruential generator $x_{n+1}=(ax_n+c)\bmod m$, $u_n=\frac{x_n}m$, was the standard source of random numbers for decades. IBM's RANDU used $a=65539$, $m=2^{31}$, $c=0$. Apply Weyl's criterion (Theorem 6.13) to the triples $(u_n,u_{n+1},u_{n+2})$: compute $\frac1N\sum_ne^{2\pi i(h_1u_n+h_2u_{n+1}+h_3u_{n+2})}$ for a few integer vectors $h$, including $(9,-6,1)$. Then explain what you see, and measure the distance between the lines carrying the pairs $(u_n,u_{n+1})$ for several classical multipliers.

**P6.5 (★) Isoperimetric ratios.** Using the FFT and Hurwitz's formulas $L^2=4\pi^2\sum n^2|c_n|^2$ and $A=\pi\sum n|c_n|^2$ (proof of Theorem 6.17), compute $\frac{4\pi A}{L^2}$ for a circle, a square and two ellipses.

## Solutions

### P6.1

In [ ]:
import numpy as np
x = np.linspace(1e-6, np.pi / 2, 200001)
def S(x, N):                                   # S_{2N-1} of the square wave
    k = np.arange(1, N + 1)[:, None]
    return 4 / np.pi * (np.sin((2 * k - 1) * x) / (2 * k - 1)).sum(axis=0)
def fejer(x, M):                               # sigma_M = sum (1 - |n|/M) c_n e_n
    n = np.arange(1, M, 2)[:, None]            # odd n < M
    return 4 / np.pi * ((1 - n / M) * np.sin(n * x) / n).sum(axis=0)
xf = np.linspace(1e-6, np.pi / 2, 4001)
for N in (5, 20, 100, 400):
    xs = x[x < 4 * np.pi / (2 * N)]              # the overshoot lives near the jump
    print(f"N = {N:3d}: max S_(2N-1) = {S(xs, N).max():.5f}   max Fejer mean sigma_(2N) = {fejer(xf, 2 * N).max():.5f}")
from mpmath import si, pi as mpi
print("limit (2/pi) Si(pi) =", float(2 / mpi * si(mpi)))

The maximum of the partial sums converges to $\frac2\pi\operatorname{Si}(\pi)=1.17898\ldots$, never to $1$, while the peak moves towards the jump. The Fejér means stay below $1$ and approach it: a non-negative kernel cannot overshoot. The price is a blurrier jump, since $\sigma_Nf$ converges more slowly than $S_Nf$ away from it. Choosing a compromise between the two is the problem of *window design* in signal processing.

### P6.2

In [ ]:
M = 2**12
t = -np.pi + 2 * np.pi * np.arange(M) / M
for name, f, exact in (("x  ", t, lambda n: 1j * (-1.0)**n / n), ("x^2", t**2, lambda n: 2 * (-1.0)**n / n**2)):
    c = np.fft.fft(f) / M * (-1.0)**np.arange(M)      # c_n for n = 0..M/2 (shift by -pi)
    errs = [abs(c[n] - exact(n)) for n in (1, 10, 100)]
    print(f"{name}: FFT coefficient errors at n = 1, 10, 100: {[f'{e:.1e}' for e in errs]}")
c2 = np.fft.fft(t**2) / M * (-1.0)**np.arange(M)
print("Parseval, sum |c_n|^2 for x^2:", (abs(c2)**2).sum(), "  (1/2pi) int x^4 =", np.pi**4 / 5)
n = np.arange(1, 10**6, dtype=float)
print("zeta(4) from the exact coefficients:", np.pi**4 / 8 * (1 / 5 - 1 / 9), " direct sum:", (1.0 / n**4).sum())

The discrete sum is a Riemann sum for $c_n$, but its exact meaning (Chapter 7) is that it equals $\sum_kc_{n+kM}$: all the frequencies congruent to $n$ modulo $M$ are folded together. This is *aliasing*. For $x$, whose periodic extension jumps, $|c_n|\approx\frac1n$ decays slowly and the folded tail is of order $\frac1M$; for $x^2$, continuous with a kink, $c_n=O(\frac1{n^2})$ and the error is of order $\frac1{M^2}$. The smoother the function, the faster its coefficients decay (Exercise 6.2), and the more accurate the FFT.

### P6.3

In [ ]:
def discrepancy(u):
    u = np.sort(u % 1.0); N = len(u); i = np.arange(1, N + 1)
    return max((i / N - u).max(), (u - (i - 1) / N).max())
N = 10**5
rng = np.random.default_rng(0)
from mpmath import mp, mpf, sqrt as msqrt, pi as mpi2, e as me, frac
mp.dps = 40
for name, a in (("golden ratio", (1 + msqrt(5)) / 2), ("sqrt 2", msqrt(2)), ("pi", mpi2), ("e", me)):
    u = np.array([float(frac(k * a)) for k in range(1, N + 1)])
    print(f"{{n {name:12s}}}: discrepancy D_N = {discrepancy(u):.2e},  N D_N = {N * discrepancy(u):6.1f}")
print(f"random points       : discrepancy D_N = {discrepancy(rng.random(N)):.2e}  (about 1/sqrt(N) = {N**-0.5:.1e})")

For the irrationals, $ND_N$ stays of order $\ln N$ or below: much more uniform than random points, whose discrepancy is of order $\frac1{\sqrt N}$ (the central limit theorem of Volume 3). The golden ratio, whose continued fraction $[1;1,1,\dots]$ has the smallest possible partial quotients, is the most uniform; $\pi$, with the large partial quotient $292$ in $[3;7,15,1,292,\dots]$, is noticeably worse. This is the idea behind *quasi-Monte Carlo* integration: replace random points by low-discrepancy sequences, and the error drops from $N^{-1/2}$ to about $\frac{\ln N}N$.

In [ ]:
from collections import Counter
first = Counter(int(str(2**k)[0]) for k in range(1, 10001))
print(" d   observed   log10(1+1/d)")
for d in range(1, 10):
    print(f" {d}   {first[d] / 10000:.4f}     {np.log10(1 + 1 / d):.4f}")

Benford's law holds to three or four decimals: the leading digit of $2^n$ is $1$ for $30.1\%$ of the exponents, as Exercise 6.5 predicts.

### P6.4

In [ ]:
def lcg(a, c, m, seed, N):
    out = np.empty(N, dtype=np.int64); x = seed
    for i in range(N):
        x = (a * x + c) % m; out[i] = x
    return out / m
N = 200000
u = lcg(65539, 0, 2**31, 1, N)                 # RANDU (IBM, 1960s)
for h in ((1, 0, 0), (1, 1, 1), (9, -6, 1), (3, -2, 5)):
    w = np.exp(2j * np.pi * (h[0] * u[:-2] + h[1] * u[1:-1] + h[2] * u[2:])).mean()
    print(f"RANDU Weyl sum at h = {str(h):12s}: |mean| = {abs(w):.4f}")
v = rng.random(N)
w = np.exp(2j * np.pi * (9 * v[:-2] - 6 * v[1:-1] + v[2:])).mean()
print(f"numpy generator at h = (9, -6, 1)   : |mean| = {abs(w):.4f}")

For a truly random sequence every Weyl sum with $h\ne0$ is of order $\frac1{\sqrt N}$. RANDU gives $1.0000$ for $h=(9,-6,1)$: the quantity $9u_n-6u_{n+1}+u_{n+2}$ is *always an integer*, so every triple of consecutive outputs lies on one of the parallel planes $9x-6y+z=k$. Only $15$ of them meet the unit cube, and simulations in three dimensions using RANDU were quietly wrong for years.

In [ ]:
print("RANDU: a^2 - 6a + 9 = 0 mod 2^31 ?", (65539**2 - 6 * 65539 + 9) % 2**31 == 0)
print(f"RANDU in 3D: h = (9, -6, 1), triples lie on planes {1 / 118**0.5:.3f} apart;"
      f" a good generator would have about {1 / (2**(1/6) * (2**31)**(1/3)):.1e}")

def shortest_dual_vector(a, m):
    """Lagrange-Gauss reduction of the lattice {(h1, h2): h1 + a h2 = 0 mod m}."""
    u, v = np.array([m, 0], dtype=object), np.array([-a, 1], dtype=object)
    norm = lambda w: w[0] * w[0] + w[1] * w[1]
    if norm(u) < norm(v): u, v = v, u
    while True:                                  # keep v the shorter vector
        q = (u[0] * v[0] + u[1] * v[1] + norm(v) // 2) // norm(v)
        u = u - q * v
        if norm(u) >= norm(v):
            return v, float(norm(v)) ** 0.5
        u, v = v, u

for a, m in ((65539, 2**31), (69069, 2**32), (1103515245, 2**31), (48271, 2**31 - 1), (3, 2**31)):
    h, nu = shortest_dual_vector(a, m)
    best = (4 / 3) ** 0.25 * m ** 0.5
    print(f"a = {a:10d}: shortest h = ({h[0]}, {h[1]}), pairs (u_n, u_n+1) lie on lines {1 / nu:.1e} apart;"
          f" ratio to the best possible: {nu / best:.3f}")

The reason is a one-line congruence: $a=2^{16}+3$, so $a^2=2^{32}+6\cdot2^{16}+9\equiv6a-9\pmod{2^{31}}$, and therefore $x_{n+2}-6x_{n+1}+9x_n\equiv0$. In general the vectors $h$ with $h_1+ah_2+a^2h_3\equiv0\pmod m$ form a lattice, and Weyl's criterion fails exactly along its short vectors. Knuth's *spectral test* computes the shortest such vector in dimensions $2$ to $6$; its length $\nu$ says the points lie on hyperplanes $\frac1\nu$ apart. In two dimensions RANDU is fine ($93\%$ of the best possible); in three it is catastrophic. A good generator must pass in every dimension that matters, which is why modern libraries use generators such as the PCG family or Mersenne Twister, and test them with batteries like TestU01.

### P6.5

In [ ]:
def deficit(xy):
    z = xy[0] + 1j * xy[1]; M = len(z)
    c = np.fft.fft(z) / M
    n = np.fft.fftfreq(M, 1 / M)
    L = 2 * np.pi * np.sqrt((n**2 * abs(c)**2).sum())     # needs arc-length parametrisation
    A = np.pi * (n * abs(c)**2).sum()
    return L, A
def arclength_resample(z, M):
    seg = np.abs(np.diff(np.append(z, z[0]))); s = np.concatenate([[0], np.cumsum(seg)])
    tgt = np.linspace(0, s[-1], M, endpoint=False)
    zz = np.append(z, z[0])
    return np.interp(tgt, s, zz.real) + 1j * np.interp(tgt, s, zz.imag), s[-1]
th = np.linspace(0, 2 * np.pi, 20000, endpoint=False)
for name, z in (("circle", np.exp(1j * th)), ("ellipse 2:1", 2 * np.cos(th) + 1j * np.sin(th)),
                ("square", None), ("ellipse 10:1", 10 * np.cos(th) + 1j * np.sin(th))):
    if z is None:
        z = np.concatenate([np.linspace(0, 1, 5000, endpoint=False) + 0j, 1 + 1j * np.linspace(0, 1, 5000, endpoint=False),
                            1 + 1j - np.linspace(0, 1, 5000, endpoint=False), 1j - 1j * np.linspace(0, 1, 5000, endpoint=False)])
    zr, Ltrue = arclength_resample(z, 4096)
    L, A = deficit((zr.real, zr.imag))
    print(f"{name:12s}: L = {L:8.4f} (true {Ltrue:8.4f}), A = {A:7.4f},  4 pi A / L^2 = {4 * np.pi * A / L**2:.4f}")

The circle reaches $\frac{4\pi A}{L^2}=1$; the square gives $\frac\pi4$, and flat ellipses approach $0$. The formulas need a parametrisation proportional to arc length, hence the resampling step. Note that the perimeter of an ellipse has no elementary closed form: Chapter 12 computes it with the arithmetic–geometric mean.